In [1]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

Loading BokehJS ...

/opt/conda/miniconda3/lib/python3.10/site-packages/hail/context.py:352: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

/opt/conda/miniconda3/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 40789
SPARKMONITOR_LISTENER: Application Started: application_1721138583160_0002 ...Start Time: 1721143341409


Running on Apache Spark version 3.3.2
SparkUI available at http://ibd-exome-m.c.daly-ibd.internal:37385
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.130-bea04d9c79b5
LOGGING: writing to /home/hail/hail-20240716-1522-0.2.130-bea04d9c79b5.log


In [2]:
Twist_mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round3/3.extract_capture_subsets/twist_capture.mt")
twist_variants = Twist_mt.rows()
Twist_mt.count()

(22285775, 97381)

In [4]:
Twist_mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'capture': str
    'sample_qc': struct {
        dp_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_hom_ref: int64, 
        n_het: int64, 
        n_hom_var: int64, 
        n_non_ref: int64, 
        n_singleton: int64, 
        n_snp: int64, 
        n_insertion: int64, 
        n_deletion: int64, 
        n_transition: int64, 
        n_transversion: int64, 
        n_star: int64, 
        r_ti_tv: float64, 
        r_het_hom_var: float64, 
        r_insertion_deletion: float64
    }
---------

In [3]:
nextera_mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round3/3.extract_capture_subsets/nextera_capture.mt")
nextera_mt = nextera_mt.filter_rows(hl.is_defined(twist_variants[nextera_mt.locus, nextera_mt.alleles]))
nextera_mt.count()

(18250584, 74703)

In [ ]:
merged_mt = Twist_mt.union_cols(nextera_mt, row_join_type='outer')
merged_mt.write("gs://ibd-exomes-gnomad-subset/QC_round3/4.join_tables/twist_nextera.mt", overwrite = True)